# Evaluating Multiple LM Outputs (External)

In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
# imports
import json
import os
import pandas as pd
import importlib.util
import sys
from os.path import join
from copy import deepcopy
from stat_genie.blade_pipeline.llms.config import llm
from stat_genie.blade_pipeline.additions.eval.extraction import \
    format_features, format_model_info
from stat_genie.blade_pipeline.additions.analysis.conclusion import \
    write_final_answer_code, make_conclusion
from stat_genie.blade_pipeline.additions.analysis.fix_code import \
    check_and_fix_code
from stat_genie.blade_pipeline.additions.eval.judge import \
    run_judge_evaluation_pairwise
from blade_bench.utils import get_dataset_info_path, get_dataset_csv_path

In [3]:
# load files
analysis_subdir_path_1 = "analysis1_output"
analysis_subdir_path_2 = "analysis2_output"
analysis_subdir_path_3 = "analysis3_output"

multirun_filename_1 = "multirun_analyses.json"
multirun_filename_2 = "multirun_analyses.json"
multirun_filename_3 = "multirun_analyses.json"

# use both files to get analysis code paths
multirun_path_1 = join(analysis_subdir_path_1, multirun_filename_1)
multirun_path_2 = join(analysis_subdir_path_2, multirun_filename_2)
multirun_path_3 = join(analysis_subdir_path_3, multirun_filename_3)

with open(multirun_path_1, "r") as file:
    multirun_analyses_1 = json.load(file)

with open(multirun_path_2, "r") as file:
    multirun_analyses_2 = json.load(file)
    
with open(multirun_path_3, "r") as file:
    multirun_analyses_3 = json.load(file)

num_analyses_1 = multirun_analyses_1['n']
num_analyses_2 = multirun_analyses_2['n']
num_analyses_3 = multirun_analyses_3['n']

analysis_code_filenames_1 = [f"llm_analysis_{i}.py" for i in range(num_analyses_1)]
analysis_code_filenames_2 = [f"llm_analysis_{i}.py" for i in range(num_analyses_2)]
analysis_code_filenames_3 = [f"llm_analysis_{i}.py" for i in range(num_analyses_3)]

analysis_code_paths_1 = [join(analysis_subdir_path_1, filename)
                         for filename in analysis_code_filenames_1]

analysis_code_paths_2 = [join(analysis_subdir_path_2, filename)
                         for filename in analysis_code_filenames_2]

analysis_code_paths_3 = [join(analysis_subdir_path_3, filename)
                         for filename in analysis_code_filenames_3]

In [4]:
# load dataset info and csv to get task and dataframe
info_path = get_dataset_info_path(multirun_analyses_1["dataset_name"])
data_path = get_dataset_csv_path(multirun_analyses_1["dataset_name"])

with open(info_path, "r") as file:
    info_json = json.load(file)
    
dataset_task = info_json["research_questions"][0]
df = pd.read_csv(data_path)

In [5]:
llm_provider = "openai"
llm_model = "gpt-5-mini"
llm_assistant = llm(provider=llm_provider, model=llm_model)

[2025-12-12 03:03:47.99][config_load.py:34 - blade_bench.llms.config_load:load_config][INFO] Loaded config from '/accounts/grad/zachrewolinski/research/stat-genie/config/llm_config.yml'.


In [6]:
features_1 = format_features(multirun_analyses_1, num_analyses_1, llm_assistant)
features_2 = format_features(multirun_analyses_2, num_analyses_2, llm_assistant)
features_3 = format_features(multirun_analyses_3, num_analyses_3, llm_assistant)

In [7]:
model_info_1 = format_model_info(multirun_analyses_1, num_analyses_1, llm_assistant)
model_info_2 = format_model_info(multirun_analyses_2, num_analyses_2, llm_assistant)
model_info_3 = format_model_info(multirun_analyses_3, num_analyses_3, llm_assistant)

In [8]:
conclusions_1 = {}

for i in range(num_analyses_1):
    # read in the txt file as a string
    conclusion_txt = os.path.abspath(os.path.join(analysis_subdir_path_1, f"final_conclusion_{i}.txt"))
    with open(conclusion_txt, "r", encoding="utf-8") as f:
        conclusion_str = f.read()
    conclusions_1[i] = conclusion_str


conclusions_2 = {}

for i in range(num_analyses_2):
    # read in the txt file as a string
    conclusion_txt = os.path.abspath(os.path.join(analysis_subdir_path_1, f"final_conclusion_{i}.txt"))
    with open(conclusion_txt, "r", encoding="utf-8") as f:
        conclusion_str = f.read()
    conclusions_2[i] = conclusion_str
    
conclusions_3 = {}

for i in range(num_analyses_3):
    # read in the txt file as a string
    conclusion_txt = os.path.abspath(os.path.join(analysis_subdir_path_1, f"final_conclusion_{i}.txt"))
    with open(conclusion_txt, "r", encoding="utf-8") as f:
        conclusion_str = f.read()
    conclusions_3[i] = conclusion_str

In [ ]:
### judge results within each group and between each group.
# within each group there should be 3 choose 2 = 3 pairwise comparisons
# between each group there should be 3 x 3 = 9 pairwise comparisons

In [15]:
features_1.keys()

dict_keys([0, 1, 2])

In [21]:
run_judge_evaluation_pairwise(dataset_task,
                              df.head(),
                              features_1,
                              features_2,
                              model_info_1,
                              model_info_2,
                              conclusions_1,
                              conclusions_2)

[2025-12-12 03:23:08.70][config_load.py:34 - blade_bench.llms.config_load:load_config][INFO] Loaded config from '/accounts/grad/zachrewolinski/research/stat-genie/config/llm_config.yml'.
[2025-12-12 03:23:09.04][config_load.py:34 - blade_bench.llms.config_load:load_config][INFO] Loaded config from '/accounts/grad/zachrewolinski/research/stat-genie/config/llm_config.yml'.
[2025-12-12 03:23:09.43][config_load.py:34 - blade_bench.llms.config_load:load_config][INFO] Loaded config from '/accounts/grad/zachrewolinski/research/stat-genie/config/llm_config.yml'.
[2025-12-12 03:23:09.69][config_load.py:34 - blade_bench.llms.config_load:load_config][INFO] Loaded config from '/accounts/grad/zachrewolinski/research/stat-genie/config/llm_config.yml'.
[2025-12-12 03:23:09.96][config_load.py:34 - blade_bench.llms.config_load:load_config][INFO] Loaded config from '/accounts/grad/zachrewolinski/research/stat-genie/config/llm_config.yml'.
[2025-12-12 03:23:10.19][config_load.py:34 - blade_bench.llms.con

{(0, 0): {'Independent Variables Similarity Score': 5,
  'Control Variables Similarity Score': 4,
  'Response Variables Similarity Score': 5,
  'Model Similarity Score': 4,
  'conclusions': 5,
  'overall_similarity': 4.6},
 (0, 1): {'Independent Variables Similarity Score': 5,
  'Control Variables Similarity Score': 4,
  'Response Variables Similarity Score': 5,
  'Model Similarity Score': 4,
  'conclusions': 5,
  'overall_similarity': 4.6},
 (0, 2): {'Independent Variables Similarity Score': 5,
  'Control Variables Similarity Score': 4,
  'Response Variables Similarity Score': 5,
  'Model Similarity Score': 4,
  'conclusions': 5,
  'overall_similarity': 4.6},
 (1, 0): {'Independent Variables Similarity Score': 5,
  'Control Variables Similarity Score': 4,
  'Response Variables Similarity Score': 5,
  'Model Similarity Score': 4,
  'conclusions': 5,
  'overall_similarity': 4.6},
 (1, 1): {'Independent Variables Similarity Score': 5,
  'Control Variables Similarity Score': 4,
  'Respons

In [ ]:
### begin with within-group performance
within_group = {1: {}, 2: {}, 3: {}}
for i in range(num_analyses_1):
    for j in range(i + 1, num_analyses_1):
        response = run_judge_evaluation_pairwise(dataset_task,
                              df.head(),
                              features_1,
                              features_2,
                              model_info_1,
                              model_info_2,
                              conclusions_1,
                              conclusions_2)
        response_dict = json.loads(response)
        within_group[1][(i, j)] = response_dict
for i in range(num_analyses_2):
    for j in range(i + 1, num_analyses_2):
        prompt = make_judge_prompt(
            dataset_task,
            data_head,
            features_2[i],
            features_2[j],
            multirun_analyses_2['analyses'][str(i)]['m_code'],
            multirun_analyses_2['analyses'][str(j)]['m_code'],
            conclusions_2[i],
            conclusions_2[j]
        )
        response = llm_judge.generate([{"role": "system",
                                         "content": judge_system_prompt},
                                        {"role": "user",
                                         "content": prompt}])
        response = response.text[0].content
        response_dict = json.loads(response)
        within_group[2][(i, j)] = response_dict
for i in range(num_analyses_3):
    for j in range(i + 1, num_analyses_3):
        prompt = make_judge_prompt(
            dataset_task,
            data_head,
            features_3[i],
            features_3[j],
            multirun_analyses_3['analyses'][str(i)]['m_code'],
            multirun_analyses_3['analyses'][str(j)]['m_code'],
            conclusions_3[i],
            conclusions_3[j]
        )
        response = llm_judge.generate([{"role": "system",
                                         "content": judge_system_prompt},
                                        {"role": "user",
                                         "content": prompt}])
        response = response.text[0].content
        response_dict = json.loads(response)
        within_group[3][(i, j)] = response_dict

In [ ]:
### now do between-group performance
between_group = { (1, 2): {}, (1, 3): {}, (2, 3): {} }
for i in range(num_analyses_1):
    for j in range(num_analyses_2):
        prompt = make_judge_prompt(
            dataset_task,
            data_head,
            features_1[i],
            features_2[j],
            multirun_analyses_1['analyses'][str(i)]['m_code'],
            multirun_analyses_2['analyses'][str(j)]['m_code'],
            conclusions_1[i],
            conclusions_2[j]
        )
        response = llm_judge.generate([{"role": "system",
                                         "content": judge_system_prompt},
                                        {"role": "user",
                                         "content": prompt}])
        response = response.text[0].content
        response_dict = json.loads(response)
        between_group[(1, 2)][(i, j)] = response_dict
for i in range(num_analyses_1):
    for j in range(num_analyses_3):
        prompt = make_judge_prompt(
            dataset_task,
            data_head,
            features_1[i],
            features_3[j],
            multirun_analyses_1['analyses'][str(i)]['m_code'],
            multirun_analyses_3['analyses'][str(j)]['m_code'],
            conclusions_1[i],
            conclusions_3[j]
        )
        response = llm_judge.generate([{"role": "system",
                                         "content": judge_system_prompt},
                                        {"role": "user",
                                         "content": prompt}])
        response = response.text[0].content
        response_dict = json.loads(response)
        between_group[(1, 3)][(i, j)] = response_dict
for i in range(num_analyses_2):
    for j in range(num_analyses_3):
        prompt = make_judge_prompt(
            dataset_task,
            data_head,
            features_2[i],
            features_3[j],
            multirun_analyses_2['analyses'][str(i)]['m_code'],
            multirun_analyses_3['analyses'][str(j)]['m_code'],
            conclusions_2[i],
            conclusions_3[j]
        )
        response = llm_judge.generate([{"role": "system",
                                         "content": judge_system_prompt},
                                        {"role": "user",
                                         "content": prompt}])
        response = response.text[0].content
        response_dict = json.loads(response)
        between_group[(2, 3)][(i, j)] = response_dict

In [ ]:
within_group

In [ ]:
within_group[3]

In [ ]:
between_group

In [ ]:
between_group[(1,2)][(1,2)]

In [ ]:
features_2[2]

In [ ]:
features_1[1]

In [ ]:
model_info_2[2]

In [ ]:
model_info_1[1]

In [ ]:
# get average similarity score for each subcategory within each group
average_within_group = {}
for group_id, comparisons in within_group.items():
    category_sums = {
        "independent_variables": 0,
        "control_variables": 0,
        "response_variables": 0,
        "model_specification": 0,
        "conclusions": 0,
        "overall_similarity": 0
    }
    num_comparisons = len(comparisons)
    
    for comparison, scores in comparisons.items():
        for category, score in scores.items():
            category_sums[category] += score
    
    average_scores = {category: total / num_comparisons
                      for category, total in category_sums.items()}
    average_within_group[group_id] = average_scores

In [ ]:
# show average within group rounded to nearest tenth
average_within_group

In [ ]:
# get average similarity score for each subcategory between each group
average_between_group = {}
for group_pair, comparisons in between_group.items():
    category_sums = {
        "independent_variables": 0,
        "control_variables": 0,
        "response_variables": 0,
        "model_specification": 0,
        "conclusions": 0,
        "overall_similarity": 0
    }
    num_comparisons = len(comparisons)
    
    for comparison, scores in comparisons.items():
        for category, score in scores.items():
            category_sums[category] += score
    
    average_scores = {category: total / num_comparisons
                      for category, total in category_sums.items()}
    average_between_group[group_pair] = average_scores

In [ ]:
average_between_group

In [ ]:
features_3